# Phase 1 - Environment Setup & Resource Verification

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
print("Torch version:", torch.__version__)
print("Torch's built-in CUDA version:", torch.version.cuda)

CUDA available: True
Device: Tesla T4
Torch version: 2.10.0+cu128
Torch's built-in CUDA version: 12.8


#### Install Libraries

In [2]:
!pip install torch_geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.5 MB/s eta 0:00:00a 0:00:01


In [3]:
!pip install rdkit --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 50.8 MB/s eta 0:00:00:00:0100:01


In [4]:
!pip install transformers PyTDC captum -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.2/154.2 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.3/151.3 kB 10.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.2/151.2 kB 10.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.2/151.2 kB 9.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.1/151.1 kB 7.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.1/151.1 kB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.1/151.1 kB 9.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.2/151.2 kB 10.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━

In [5]:
## numpy pillow check
import rdkit
import torch_geometric
import captum
import transformers
from tdc.single_pred import Tox
import numpy as np
 
print("rdkit:", rdkit.__version__)
print("torch_geometric:", torch_geometric.__version__)
print("captum:", captum.__version__)
print("transformers:", transformers.__version__)
print("numpy:", np.__version__)

rdkit: 2026.03.6
torch_geometric: 2.8.0.post1
captum: 0.9.0
transformers: 5.0.0
numpy: 2.0.2


#### AMES Toxicity Dataset (from TDC)

In [6]:
from tdc.single_pred import Tox

data = Tox(name='AMES')
df = data.get_data(format='df')
print(df.shape)
print(df.head())

Downloading...
100%|██████████| 344k/344k [00:00<00:00, 3.28MiB/s]
Loading...
Done!


(7278, 3)
  Drug_ID                                               Drug  Y
0  Drug 0  O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...  1
1  Drug 1       O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2  1
2  Drug 2  O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...  0
3  Drug 3                          [N-]=[N+]=CC(=O)NCC(=O)NN  1
4  Drug 4                          [N-]=[N+]=C1C=NC(=O)NC1=O  1


#### Scaffold Split

In [7]:
split = data.get_split(method = "scaffold", seed = 42, frac = [0.7, 0.1, 0.2])
train, valid, test = split['train'], split['valid'], split['test']
print(f"train = {len(train)}  valid = {len(valid)}  test = {len(test)}")

100%|██████████| 7278/7278 [00:01<00:00, 4422.20it/s]

train = 5094  valid = 727  test = 1457


In [8]:
## And to catch any broken SMILES, we will run RDKit over the dataset.
from rdkit import Chem

def is_valid(smiles):
    return Chem.MolFromSmiles(smiles) is not None

bad = df[~df['Drug'].apply(is_valid)]
print(f"Unparseable SMILES: {len(bad)} / {len(df)}")

Unparseable SMILES: 0 / 7278


#### Sequence Model and Tokenizer

In [9]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

seq_ckpt = "seyonec/ChemBERTa-zinc-base-v1"
tokenizer = AutoTokenizer.from_pretrained(seq_ckpt)
seq_model = AutoModelForSequenceClassification.from_pretrained(seq_ckpt, num_labels = 2)
## quick forward pass to test 
example_smiles = df['Drug'].iloc[0]
tokens = tokenizer(example_smiles, return_tensors = "pt")
print(tokens)
print(seq_model(**tokens).logits)

config.json:   0%|          | 0.00/501 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: seyonec/ChemBERTa-zinc-base-v1
Key                         | Status     | 
----------------------------+------------+-
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'input_ids': tensor([[  0,  51, 361,  50, 368,  51, 292,  71,  21, 264,  22, 264,  23, 264,
         284,  50, 306,  51, 272,  51, 292,  71,  24,  71,  25, 269,  25,  71,
          21,  71,  22,  71, 383,   2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


model.safetensors:   0%|          | 0.00/179M [00:00<?, ?B/s]

tensor([[-0.0426, -0.1311]], grad_fn=<AddmmBackward0>)


In [10]:
## verify libraries and class balance
import torch_geometric
import captum
print("torch_geometric:", torch_geometric.__version__)
print("captum:", captum.__version__)

print("Train label balance:\n", train['Y'].value_counts(normalize = True))
print("Test label balance:\n", test['Y'].value_counts(normalize = True))

torch_geometric: 2.8.0.post1
captum: 0.9.0
Train label balance:
 Y
1    0.528661
0    0.471339
Name: proportion, dtype: float64
Test label balance:
 Y
1    0.597117
0    0.402883
Name: proportion, dtype: float64


# Phase 2A - RDkit Standardization

In [11]:
from rdkit import Chem
from rdkit.Chem.SaltRemover import SaltRemover

remover = SaltRemover()

def standardize_smiles(smiles):
    """Cleans up a SMILES string: strips salts, sanitizes, and returns the canonical version. Returns None if parsing fails."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = remover.StripMol(mol, dontRemoveEverything=True)
    if mol is None or mol.GetNumAtoms() == 0:
        return None
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        return None
    return Chem.MolToSmiles(mol, canonical=True)

# Check for multi-fragment molecules (salts/counterions containing '.')
multi_fragment = df['Drug'].apply(lambda s: '.' in s)
print(f"Multi-fragment entries: {multi_fragment.sum()} / {len(df)}")

# Run the standardization pipeline across the whole dataset
df['Drug_standardized'] = df['Drug'].apply(standardize_smiles)

# Track failures and modifications
failed = df[df['Drug_standardized'].isna()]
print(f"Failed to standardize: {len(failed)} / {len(df)}")

changed = df[(df['Drug_standardized'].notna()) & (df['Drug_standardized'] != df['Drug'])]
print(f"Changed by standardization (salt stripped / re-canonicalized): {len(changed)} / {len(df)}")

# Visual check of a few modified molecules
print(changed[['Drug', 'Drug_standardized']].head(10))

# Drop any broken rows and finalize the clean dataset
df_clean = df[df['Drug_standardized'].notna()].copy()
print(f"Final clean dataset: {len(df_clean)} / {len(df)}")

Multi-fragment entries: 0 / 7278
Failed to standardize: 0 / 7278
Changed by standardization (salt stripped / re-canonicalized): 44 / 7278
                                                   Drug  \
33     CC(C)[C@@H]1CC[C@H](C)[C@@H]2CC[C@H](C)C[C@@H]12   
631   ClC1=C(Cl)[C@]2(Cl)[C@@H]3[C@@H](Cl)C=C[C@H]3[...   
677              C(=C/c1cccc(/C=C/c2ccccc2)c1)\c1ccccc1   
837                       C[C@H]1CN(N=O)C[C@@H](C)N1N=O   
852        CC1=C(/C=C\C(C)=C/C=C\C(C)=C/C=O)C(C)(C)CCC1   
965       N=c1ccn2c(n1)O[C@@H]1[C@@H]2O[C@H](CO)[C@H]1O   
1016              OC[C@H](O)[C@@H](O)[C@H](O)[C@H](O)CO   
1046                C1O[C@H]1[C@@H]1CC[C@@H]2O[C@@H]2C1   
1162  C[C@]12CC(C=O)=C(O)C[C@@H]1CC[C@@H]1[C@@H]2CC[...   
1202  CC[C@H](CC[C@@H](C)[C@H]1CC[C@H]2[C@@H]3CC=C4C...   

                                      Drug_standardized  
33      CC(C)[C@@H]1CC[C@H](C)[C@@H]2CC[C@H](C)C[C@H]21  
631   ClC1=C(Cl)[C@]2(Cl)[C@H]3[C@@H](C=C[C@@H]3Cl)[...  
677              C(=C\c1cccc(/C=C/c2cc

# Phase 2B - Rao et al. Ground-truth Atom Masks Aligned to TDC AMS Molecules

In [12]:
import numpy as np
from rdkit import Chem

# Load ground-truth attributions
gt_data = np.load("/kaggle/input/datasets/minoola33/mutagenicity-groundtruth-attributions/attributions.npz", allow_pickle=True)["attributions"]
print(f"Rao et al. ground-truth molecules: {len(gt_data)}")

# Build a lookup dictionary keyed by canonical SMILES
gt_lookup = {}
parse_failures = 0

for item in gt_data:
    raw_smiles = item["SMILES"].strip()
    mol = Chem.MolFromSmiles(raw_smiles)
    if mol is None:
        parse_failures += 1
        continue
    canon = Chem.MolToSmiles(mol, canonical=True)
    gt_lookup[canon] = {
        "mol": mol,            # Atom order matches node_atts
        "node_atts": item["node_atts"],
        "label": item["label"],
    }

print(f"Parse failures in ground-truth set: {parse_failures}")
print(f"Unique canonical molecules in ground-truth lookup: {len(gt_lookup)}")

# Match against our clean TDC dataset (df_clean must already have 'Drug_standardized')
matched, unmatched = [], []

for _, row in df_clean.iterrows():
    canon = row['Drug_standardized']
    if canon in gt_lookup:
        matched.append(row['Drug_ID'])
    else:
        unmatched.append(row['Drug_ID'])

print(f"Matched to ground truth: {len(matched)} / {len(df_clean)}")
print(f"Unmatched (no ground-truth coverage): {len(unmatched)}")

# Function to safely reorder ground-truth atom masks to match our SMILES parse order
def align_ground_truth(our_smiles_canon):
    if our_smiles_canon not in gt_lookup:
        return None

    entry = gt_lookup[our_smiles_canon]
    their_mol = entry["mol"]
    their_atts = entry["node_atts"]

    our_mol = Chem.MolFromSmiles(our_smiles_canon)

    if their_mol.GetNumAtoms() != our_mol.GetNumAtoms():
        return None

    match = our_mol.GetSubstructMatch(their_mol)
    if not match or len(match) != our_mol.GetNumAtoms():
        return None

    aligned = np.zeros(our_mol.GetNumAtoms(), dtype=int)
    for their_idx, our_idx in enumerate(match):
        aligned[our_idx] = their_atts[their_idx]
    return aligned

# Sanity check alignment across matched molecules
sample_checked = 0
alignment_failures = 0
for _, row in df_clean.iterrows():
    if row['Drug_ID'] not in matched:
        continue
    mask = align_ground_truth(row['Drug_standardized'])
    if mask is None:
        alignment_failures += 1
    sample_checked += 1
    if sample_checked >= len(matched):
        break

print(f"Alignment failures among matched molecules: {alignment_failures} / {len(matched)}")

# Cross-check label consistency between Rao et al. and TDC
label_mismatches = 0
for _, row in df_clean.iterrows():
    canon = row['Drug_standardized']
    if canon in gt_lookup:
        if gt_lookup[canon]["label"] != row['Y']:
            label_mismatches += 1

print(f"Label disagreements between Rao et al. and TDC: {label_mismatches} / {len(matched)}")

Rao et al. ground-truth molecules: 6506
Parse failures in ground-truth set: 0
Unique canonical molecules in ground-truth lookup: 6505
Matched to ground truth: 6426 / 7278
Unmatched (no ground-truth coverage): 852
Alignment failures among matched molecules: 0 / 6426
Label disagreements between Rao et al. and TDC: 4 / 6426


#### Test Coverage Check

In [13]:
import numpy as np
from rdkit import Chem

# test = the Phase 1 locked test split (Drug_ID, Drug, Y)
test_ids = set(test['Drug_ID'])
matched_set = set(matched)

test_matched = test_ids & matched_set
test_unmatched = test_ids - matched_set

print(f"Test set ground-truth coverage: {len(test_matched)} / {len(test_ids)} "
      f"({100*len(test_matched)/len(test_ids):.1f}%)")

# Is the unmatched set systematically different? (e.g., are we silently losing all the largest or smallest molecules?)
def atom_count(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return mol.GetNumAtoms() if mol else None

df_clean['atom_count'] = df_clean['Drug_standardized'].apply(atom_count)
matched_sizes = df_clean[df_clean['Drug_ID'].isin(matched_set)]['atom_count']
unmatched_sizes = df_clean[~df_clean['Drug_ID'].isin(matched_set)]['atom_count']

print(f"Matched  molecules -- mean atoms: {matched_sizes.mean():.1f}, median: {matched_sizes.median():.0f}")
print(f"Unmatched molecules -- mean atoms: {unmatched_sizes.mean():.1f}, median: {unmatched_sizes.median():.0f}")

Test set ground-truth coverage: 1217 / 1457 (83.5%)
Matched  molecules -- mean atoms: 16.7, median: 16
Unmatched molecules -- mean atoms: 17.4, median: 17


# Phase 3 - Model Training

#### GPU Setup

In [14]:
import rdkit
import torch_geometric
import captum
import transformers
from tdc.single_pred import Tox

print("rdkit:", rdkit.__version__)
print("torch_geometric:", torch_geometric.__version__)
print("captum:", captum.__version__)
print("transformers:", transformers.__version__)

rdkit: 2026.03.6
torch_geometric: 2.8.0.post1
captum: 0.9.0
transformers: 5.0.0


#### Data Reload & Split Alignment

In [14]:
from tdc.single_pred import Tox
from rdkit import Chem
from rdkit.Chem.SaltRemover import SaltRemover

data = Tox(name='AMES')
df = data.get_data(format='df')
split = data.get_split(method="scaffold", seed=42, frac=[0.7, 0.1, 0.2])
train_df, valid_df, test_df = split['train'], split['valid'], split['test']

# Standardize SMILES strings (salt stripping + canonicalization)
remover = SaltRemover()

def standardize_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = remover.StripMol(mol, dontRemoveEverything=True)
    if mol is None or mol.GetNumAtoms() == 0:
        return None
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        return None
    return Chem.MolToSmiles(mol, canonical=True)

df['Drug_standardized'] = df['Drug'].apply(standardize_smiles)
df_clean = df[df['Drug_standardized'].notna()].copy()

# Map original split assignments back using Drug_IDs (to filter)
train_ids = set(train_df['Drug_ID'])
valid_ids = set(valid_df['Drug_ID'])
test_ids = set(test_df['Drug_ID'])

df_clean['split'] = df_clean['Drug_ID'].apply(
    lambda i: 'train' if i in train_ids else ('valid' if i in valid_ids else ('test' if i in test_ids else 'unknown'))
)

print(df_clean['split'].value_counts())

# Ensure no molecules slipped through without a split assignment
assert (df_clean['split'] == 'unknown').sum() == 0, "Some molecules lost their split membership -- check filtering."

Found local copy...
Loading...
Done!
100%|██████████| 7278/7278 [00:01<00:00, 4489.59it/s]


split
train    5094
test     1457
valid     727
Name: count, dtype: int64


#### Fine-tuning ChemBERTa

In [15]:
# =========================================================
# PHASE 3 — ChemBERTa fine-tuning
# Run phase3_kaggle_setup.py and phase3_data_reload.py first.
# =========================================================

import torch
import numpy as np
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score

device = "cuda" if torch.cuda.is_available() else "cpu"
CKPT = "seyonec/ChemBERTa-zinc-base-v1"

from transformers import logging as hf_logging

tokenizer = AutoTokenizer.from_pretrained(CKPT)
hf_logging.set_verbosity_error()  # Suppress expected legacy-naming warnings
model = AutoModelForSequenceClassification.from_pretrained(CKPT, num_labels=2).to(device)
hf_logging.set_verbosity_warning()  # Restore normal warnings

# Load directly from pytorch_model.bin to bypass any safetensors mapping discrepancies 
# and properly pull the full encoder state into the model.
from huggingface_hub import hf_hub_download

ckpt_file = hf_hub_download(repo_id=CKPT, filename="pytorch_model.bin")
raw_state = torch.load(ckpt_file, map_location="cpu")

model_state = model.state_dict()
matched_keys = [k for k in raw_state if k in model_state]
print(f"{len(matched_keys)} / {len(raw_state)} raw checkpoint keys match the model's "
      f"keys directly (no renaming needed -- this file uses current naming).")

with torch.no_grad():
    for k in matched_keys:
        model_state[k].copy_(raw_state[k])

print(f"Restored {len(matched_keys)} pretrained parameters directly from the verified checkpoint.")

# Check which roberta parameters were not restored (pooler keys are normal to skip here).
model_roberta_keys = [k for k in model_state if k.startswith("roberta.")]
still_unrestored = [k for k in model_roberta_keys if k not in matched_keys]
print(f"roberta.* parameters NOT restored from the checkpoint: {len(still_unrestored)}")
for k in still_unrestored:
    print(" ", k)

class SMILESDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.smiles = df['Drug_standardized'].tolist()
        self.labels = df['Y'].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.smiles[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = SMILESDataset(df_clean[df_clean['split'] == 'train'], tokenizer)
valid_ds = SMILESDataset(df_clean[df_clean['split'] == 'valid'], tokenizer)
test_ds = SMILESDataset(df_clean[df_clean['split'] == 'test'], tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)
    return {
        "auroc": roc_auc_score(labels, probs),
        "auprc": average_precision_score(labels, probs),
        "accuracy": accuracy_score(labels, preds),
    }

args = TrainingArguments(
    output_dir="/kaggle/working/chemberta_ames",
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="auroc",
    greater_is_better=True,
    save_total_limit=1,  # Keep only the current best checkpoint on disk
    logging_steps=20,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

# Final evaluation on the held-out test set
test_results = trainer.evaluate(test_ds)
print("ChemBERTa TEST results:", test_results)

# Save the fine-tuned model for reuse in Phase 4/5
trainer.save_model("/kaggle/working/chemberta_ames_final")
tokenizer.save_pretrained("/kaggle/working/chemberta_ames_final")
print("Saved to /kaggle/working/chemberta_ames_final")

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

101 / 110 raw checkpoint keys match the model's keys directly (no renaming needed -- this file uses current naming).
Restored 101 pretrained parameters directly from the verified checkpoint.
roberta.* parameters NOT restored from the checkpoint: 0


Epoch,Training Loss,Validation Loss,Auroc,Auprc,Accuracy
1,1.095140,1.043499,0.817441,0.836692,0.762036
2,1.003975,1.043772,0.823308,0.845973,0.763411
3,0.887922,1.045279,0.831224,0.851920,0.762036
4,0.828049,1.066560,0.833834,0.850290,0.767538
5,0.766845,1.055958,0.838785,0.860790,0.785420
6,0.663601,1.071186,0.843543,0.868787,0.785420
7,0.628179,1.097111,0.844790,0.869893,0.767538
8,0.569479,1.084841,0.849279,0.873247,0.778542
9,0.577170,1.114351,0.848294,0.870117,0.777166
10,0.530237,1.116297,0.848956,0.871823,0.782669


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

ChemBERTa TEST results: {'eval_loss': 1.2202093601226807, 'eval_auroc': 0.8068358495369011, 'eval_auprc': 0.8558117822447564, 'eval_accuracy': 0.7446808510638298, 'eval_runtime': 3.0632, 'eval_samples_per_second': 475.651, 'eval_steps_per_second': 3.918, 'epoch': 10.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /kaggle/working/chemberta_ames_final


In [16]:
import shutil
shutil.make_archive("/kaggle/working/chemberta_ames_final", 'zip', "/kaggle/working/chemberta_ames_final")
print("Zipped to /kaggle/working/chemberta_ames_final.zip")

Zipped to /kaggle/working/chemberta_ames_final.zip


#### GIN Training

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool
from rdkit import Chem
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

# Convert an RDKit molecule into a PyTorch Geometric graph object
def mol_to_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0:
        return None

    atom_feats = []
    for atom in mol.GetAtoms():
        atom_feats.append([
            atom.GetAtomicNum(),
            atom.GetDegree(),
            atom.GetFormalCharge(),
            int(atom.GetHybridization()),
            int(atom.GetIsAromatic()),
            atom.GetTotalNumHs(),
        ])
    x = torch.tensor(atom_feats, dtype=torch.float)

    edge_index = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index += [[i, j], [j, i]]
        
    if len(edge_index) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    y = torch.tensor([label], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, y=y)

def build_dataset(df_subset):
    data_list = []
    for _, row in df_subset.iterrows():
        d = mol_to_data(row['Drug_standardized'], row['Y'])
        if d is not None:
            data_list.append(d)
    return data_list

# Build datasets and data loaders for each split
train_data = build_dataset(df_clean[df_clean['split'] == 'train'])
valid_data = build_dataset(df_clean[df_clean['split'] == 'valid'])
test_data = build_dataset(df_clean[df_clean['split'] == 'test'])

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=128, shuffle=False)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

# Define the Graph Isomorphism Network (GIN) architecture
class GIN(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, num_layers=3, dropout=0.2):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        for i in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(in_dim if i == 0 else hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINConv(mlp))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
            
        self.dropout = dropout
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index, batch):
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_add_pool(x, batch)
        return self.classifier(x).squeeze(-1)

model = GIN(in_dim=6).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss()

# Evaluation helper function
def evaluate(loader):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch.x, batch.edge_index, batch.batch)
            all_logits.append(logits.cpu())
            all_labels.append(batch.y.cpu())
            
    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy().ravel()
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    
    return {
        "auroc": roc_auc_score(labels, probs),
        "auprc": average_precision_score(labels, probs),
        "accuracy": accuracy_score(labels, preds),
    }

best_val_auroc = 0
best_state = None
epochs_since_improvement = 0
PATIENCE = 15  # Stop early if validation AUROC doesn't improve for 15 epochs

# Training loop
for epoch in range(1, 101):
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        logits = model(batch.x, batch.edge_index, batch.batch)
        
        # Ensure target shape matches logits
        loss = criterion(logits, batch.y.view(-1))
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs

    val_metrics = evaluate(valid_loader)
    
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch}: train_loss={total_loss/len(train_data):.4f}  "
              f"val_auroc={val_metrics['auroc']:.4f}  val_auprc={val_metrics['auprc']:.4f}")

    # Track best model and manage early stopping counter
    if val_metrics['auroc'] > best_val_auroc:
        best_val_auroc = val_metrics['auroc']
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_since_improvement = 0
    else:
        epochs_since_improvement += 1
        if epochs_since_improvement >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch} (no improvement for {PATIENCE} epochs). "
                  f"Best val_auroc={best_val_auroc:.4f}")
            break

# Load best checkpoint and evaluate on the test set
model.load_state_dict(best_state)
test_metrics = evaluate(test_loader)
print("GIN TEST results:", test_metrics)

# Save the final model weights
torch.save(best_state, "/kaggle/working/gin_ames_final.pt")
print("Saved checkpoint to /kaggle/working/gin_ames_final.pt")

Epoch 1: train_loss=0.7453  val_auroc=0.7557  val_auprc=0.7789
Epoch 5: train_loss=0.5967  val_auroc=0.7080  val_auprc=0.7194
Epoch 10: train_loss=0.5624  val_auroc=0.7736  val_auprc=0.7951
Epoch 15: train_loss=0.5335  val_auroc=0.8143  val_auprc=0.8425
Epoch 20: train_loss=0.5222  val_auroc=0.8288  val_auprc=0.8430
Epoch 25: train_loss=0.5074  val_auroc=0.8300  val_auprc=0.8414
Epoch 30: train_loss=0.5039  val_auroc=0.8121  val_auprc=0.8356
Epoch 35: train_loss=0.4910  val_auroc=0.8335  val_auprc=0.8372
Epoch 40: train_loss=0.4820  val_auroc=0.8268  val_auprc=0.8528
Epoch 45: train_loss=0.4782  val_auroc=0.8236  val_auprc=0.8405
Epoch 50: train_loss=0.4698  val_auroc=0.8250  val_auprc=0.8389
Early stopping triggered at epoch 52 (no improvement for 15 epochs). Best val_auroc=0.8348
GIN TEST results: {'auroc': np.float64(0.7844915702285142), 'auprc': np.float64(0.8322445414027896), 'accuracy': 0.6787920384351407}
Saved checkpoint to /kaggle/working/gin_ames_final.pt


In [17]:
!ls -la /kaggle/working/
!ls -la /kaggle/working/chemberta_ames_final 2>/dev/null || echo "NOT FOUND"
!ls -la /kaggle/working/chemberta_ames_final.zip 2>/dev/null || echo "ZIP NOT FOUND"

total 160040
drwxr-xr-x 6 root root      4096 Sep 14 07:01 .
drwxr-xr-x 5 root root      4096 Sep 14 06:51 ..
drwxr-xr-x 3 root root      4096 Sep 14 06:59 chemberta_ames
drwxr-xr-x 2 root root      4096 Sep 14 06:59 chemberta_ames_final
-rw-r--r-- 1 root root 163852907 Sep 14 07:01 chemberta_ames_final.zip
drwxr-xr-x 2 root root      4096 Sep 14 06:53 data
drwxr-xr-x 2 root root      4096 Sep 14 06:52 .virtual_documents
total 172364
drwxr-xr-x 2 root root      4096 Sep 14 06:59 .
drwxr-xr-x 6 root root      4096 Sep 14 07:01 ..
-rw-r--r-- 1 root root       759 Sep 14 06:59 config.json
-rw-r--r-- 1 root root 176434280 Sep 14 06:59 model.safetensors
-rw-r--r-- 1 root root       377 Sep 14 06:59 tokenizer_config.json
-rw-r--r-- 1 root root     38442 Sep 14 06:59 tokenizer.json
-rw-r--r-- 1 root root      5201 Sep 14 06:59 training_args.bin
-rw-r--r-- 1 root root 163852907 Sep 14 07:01 /kaggle/working/chemberta_ames_final.zip


In [18]:
from IPython.display import FileLink
FileLink('chemberta_ames_final.zip')

/kaggle/working/chemberta_ames_final.zip

In [20]:
import shutil
shutil.make_archive('/kaggle/working/gin_ames_final', 'zip', '/kaggle/working', 'gin_ames_final.pt')

from IPython.display import FileLink
import os
os.chdir('/kaggle/working')
FileLink('gin_ames_final.zip')

/kaggle/working/gin_ames_final.zip